# LTX Director 2.0 — Colab G4 精简启动脚本

适配工作流：`LTX_Director_2_Workflow_Distilled.json`　运行时：**G4 高 RAM**（RTX PRO 6000 Blackwell / 96GB）

本版本已根据工作流 JSON 逐节点核对，用不到的节点仓库与模型全部注释保留（没有删除，随时可恢复）。

**改动摘要**

| 项 | 原脚本 | 现在 |
| --- | --- | --- |
| 节点仓库 | 4 个（含 PromptRelay）+ rgthree | 3 个：LTXVideo / KJNodes / WhatDreamsCost |
| 主模型 | 22b-dev-fp8 完整 checkpoint | distilled transformer fp8（UNETLoader 实际引用） |
| 蒸馏 LoRA | 下载 | 注释（工作流无 LoraLoader） |
| Fish Audio S2 | 下载 | 注释（工作流无 TTS 节点） |
| Google API 补丁 | 执行 | 注释（本地 gemma 编码器，不调用云端） |
| 模型数量 | 9 个 | 6 个 |

按顺序执行 Cell 1 → 8 即可。


In [ ]:
# ==========================================
# Cell 1: 基础环境 (ComfyUI 本体)
# 本 Notebook 已严格按 LTX_Director_2_Workflow_Distilled.json 精简
# 目标运行时: Colab "G4 高 RAM" (NVIDIA RTX PRO 6000 Blackwell, 96GB VRAM)
# ==========================================
import os, subprocess
print("=== 🚀 开始安装 ComfyUI 本体 ===")
%cd /content

if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI
else:
    !cd ComfyUI && git pull

%cd /content/ComfyUI
!pip install -q -r requirements.txt huggingface_hub hf_transfer

# --- Blackwell (sm_120) 自检: 确认 torch 没被 requirements.txt 降级成不支持的旧版 ---
import torch
print("torch:", torch.__version__, "| cuda:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| capability:", torch.cuda.get_device_capability(0))
    if torch.cuda.get_device_capability(0)[0] >= 12:
        print("✅ 检测到 Blackwell, fp8 / fp4 权重可原生运行")
else:
    print("⚠️ 未检测到 GPU, 请检查运行时类型")

# --- 通用插件: 工作流本身不需要, 仅 Manager 便于排查缺失节点 ---
def install_node(repo_url):
    folder_name = repo_url.split('/')[-1].replace('.git', '')
    target_path = f"/content/ComfyUI/custom_nodes/{folder_name}"
    if not os.path.exists(target_path):
        !git clone {repo_url} {target_path}
        if os.path.exists(f"{target_path}/requirements.txt"):
            !pip install -q -r {target_path}/requirements.txt
    else:
        !cd {target_path} && git pull

base_nodes = [
    "https://github.com/ltdrdata/ComfyUI-Manager.git",     # 可选, 排错用
    # "https://github.com/rgthree/rgthree-comfy.git",      # ❌ 工作流未使用, 已注释
]
for node in base_nodes:
    install_node(node)

print("\n✅ Cell 1 完成")


In [ ]:
# ==========================================
# Cell 2: 自定义节点安装 (只装工作流实际用到的 3 个仓库)
# ------------------------------------------
# 工作流内 33 个节点的归属核对结果:
#   ComfyUI 核心自带 : UNETLoader / DualCLIPLoader / VAELoader / VAEDecode /
#                      RandomNoise / KSamplerSelect / BasicScheduler / CFGGuider /
#                      SamplerCustomAdvanced / ConditioningZeroOut /
#                      LatentUpscaleModelLoader / CreateVideo / SaveVideo / MarkdownNote
#   ComfyUI-LTXVideo : LTXVConditioning / LTXVLatentUpsampler / LTXVAudioVAEDecode /
#                      LTXVConcatAVLatent / LTXVSeparateAVLatent
#   ComfyUI-KJNodes  : VAELoaderKJ / ModelPreviewOverrideKJ
#   WhatDreamsCost   : LTXDirector / LTXDirectorGuide / LTXDirectorCropGuides
# ==========================================
import os
import subprocess

comfyui_dir = "/content/ComfyUI"
custom_nodes_dir = os.path.join(comfyui_dir, "custom_nodes")

node_repos = [
    # 核心 LTX 视频节点 (必选: LTXVConditioning / LTXVLatentUpsampler /
    # LTXVAudioVAEDecode / LTXVConcatAVLatent / LTXVSeparateAVLatent)
    "https://github.com/Lightricks/ComfyUI-LTXVideo.git",

    # KJNodes (必选: VAELoaderKJ 加载 tiny/video/audio VAE, ModelPreviewOverrideKJ 采样预览)
    "https://github.com/kijai/ComfyUI-KJNodes.git",

    # 本工作流主体 (必选: LTXDirector / LTXDirectorGuide / LTXDirectorCropGuides)
    "https://github.com/WhatDreamsCost/WhatDreamsCost-ComfyUI.git",

    # ==========================================
    # 以下为本工作流未用到的节点, 已注释
    # ==========================================
    # ❌ PromptRelay: Prompt Relay 逻辑已内置在 WhatDreamsCost 仓库的 prompt_relay.py 中,
    #    工作流没有任何独立的 PromptRelay 节点, 无需再装
    # "https://github.com/kijai/ComfyUI-PromptRelay.git",
    # ❌ 视频存取用的是核心 CreateVideo / SaveVideo, 不需要 VHS
    # "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    # "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
    # "https://github.com/cubiq/ComfyUI_essentials.git",
    # "https://github.com/yuvraj108c/ComfyUI-Video-Depth-Anything.git",
    # "https://github.com/evanspearman/ComfyMath.git",
    # "https://github.com/chrisgoringe/cg-use-everywhere.git",
    # "https://github.com/rgthree/rgthree-comfy.git",
    # "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
    # "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git",
    # "https://github.com/Suzie1/ComfyUI_Comfyroll_CustomNodes.git",
    # "https://github.com/WASasquatch/was-node-suite-comfyui.git",
    # "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git",
    # "https://github.com/DoctorDiffusion/ComfyUI-MediaMixer.git",
    # "https://github.com/ltdrdata/ComfyUI-Inspire-Pack.git",
    # "https://github.com/11cafe/comfyui-workspace-manager.git",
    # "https://github.com/Saganaki22/ComfyUI-FishAudioS2.git",
    # "https://github.com/chflame163/ComfyUI_LayerStyle.git",
    # "https://github.com/yolain/ComfyUI-Easy-Use.git",
    # "https://github.com/BadCafeCode/masquerade-nodes-comfyui.git",
    # "https://github.com/ClownsharkBatwing/RES4LYF.git",
    # "https://github.com/GoogleCloudPlatform/comfyui-google-genmedia-custom-nodes.git",
]

print("=== 🚀 开始安装工作流所需节点及依赖 ===")
os.makedirs(custom_nodes_dir, exist_ok=True)

for repo in node_repos:
    repo_name = repo.split('/')[-1].replace('.git', '')
    repo_path = os.path.join(custom_nodes_dir, repo_name)

    if not os.path.exists(repo_path):
        print(f"\n📥 正在克隆: {repo_name}...")
        result = subprocess.run(["git", "clone", repo, repo_path],
                                capture_output=True, text=True)
        if result.returncode != 0:
            print(f"   ❌ 克隆失败: {result.stderr}")
        else:
            print("   ✅ 克隆成功")
    else:
        print(f"\n✅ 已存在: {repo_name}, 正在尝试拉取更新...")
        pull_result = subprocess.run(["git", "pull"], cwd=repo_path,
                                     capture_output=True, text=True)
        if pull_result.returncode != 0:
            print(f"   ⚠️ 更新失败: {pull_result.stderr.strip()}")
        else:
            print("   🔄 代码已是最新或更新成功")

    req_file = os.path.join(repo_path, "requirements.txt")
    if os.path.exists(req_file):
        print(f"   ⚙️ 安装 {repo_name} 的依赖...")
        subprocess.run(["pip", "install", "-r", req_file, "--quiet"])

# ==========================================
# 🔧 特定依赖修复
# ==========================================
print("\n=== 🔧 执行依赖修复 ===")
# ComfyUI-LTXVideo 对 kornia 版本敏感
subprocess.run(["pip", "install", "kornia==0.7.2", "--quiet"])
# 防止上面各 requirements.txt 把 torch 降级 (Blackwell 需要 cu128+ 的 torch)
import torch
print("修复后 torch:", torch.__version__, "| cuda:", torch.version.cuda)

print("\n🎉 节点安装完成")


In [ ]:
# ==========================================
# Cell 3: Fish Audio S2 (s2-pro-fp8) —— ❌ 本工作流完全未使用, 整段已注释
# ------------------------------------------
# 原因: LTX_Director_2_Workflow_Distilled.json 中没有 FishS2MultiSpeakerTTS 等节点,
#       音频由 LTXVAudioVAEDecode + LTX23_audio_vae 直接生成, 不需要外挂 TTS 模型。
#       该模型约数 GB, 跳过可显著缩短启动时间。
# 如确实需要 TTS, 请同时取消 Cell 2 里 ComfyUI-FishAudioS2 的注释。
# ==========================================

# import os
# from huggingface_hub import snapshot_download
#
# comfyui_dir = "/content/ComfyUI"
# target_dir = os.path.join(comfyui_dir, "models", "FishAudioS2", "s2-pro-fp8")
#
# print(f"🚀 开始下载 s2-pro-fp8 模型...")
# print(f"📂 目标安装路径: {target_dir}")
#
# try:
#     os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
#     snapshot_download(
#         repo_id="drbaph/s2-pro-fp8",
#         local_dir=target_dir,
#         local_dir_use_symlinks=False
#     )
#     print("\n🎉 s2-pro-fp8 模型下载并安装成功！")
# except Exception as e:
#     print(f"\n❌ 下载失败, 错误信息: {e}")

print("⏩ 已跳过 Fish Audio S2 (本工作流不需要)")


In [ ]:
# ==========================================
# Cell 4: 模型下载 (严格按工作流内 Loader 节点的 widget 值核对)
# ------------------------------------------
# 工作流实际引用的文件名 (从 JSON 里逐个读出):
#   UNETLoader             -> ltx-2.3-22b-distilled-1.1_transformer_only_fp8_scaled.safetensors
#   DualCLIPLoader         -> gemma_3_12B_it_fp4_mixed.safetensors + ltx-2.3_text_projection_bf16.safetensors
#   VAELoader   (视频)     -> LTX23_video_vae_bf16.safetensors
#   VAELoader   (音频)     -> LTX23_audio_vae_bf16.safetensors
#   VAELoaderKJ (快速预览) -> taeltx2_3.safetensors
#   LatentUpscaleModelLoader -> ltx-2.3-spatial-upscaler-x2-1.1.safetensors
# 共 6 个文件。工作流里没有 CheckpointLoaderSimple, 也没有任何 LoraLoader,
# 所以原脚本里的 22b-dev-fp8 大套餐和蒸馏 LoRA 都不需要下载。
# ==========================================
import os
import shutil
from huggingface_hub import hf_hub_download
from concurrent.futures import ThreadPoolExecutor

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
except Exception:
    print("⚠️ 未读到 HF_TOKEN (左侧 🔑 Secrets), 公开仓库仍可下载, 受限仓库会失败")

# UNETLoader 在新版 ComfyUI 读 models/diffusion_models, 旧版读 models/unet
# 下面两个目录都会放一份 (硬链接, 不占额外空间), 避免版本差异导致找不到模型
UNET_DIR = "/content/ComfyUI/models/diffusion_models"

downloads = [
    # --- 1. 主模型: 蒸馏版 transformer (UNETLoader) ---
    {"repo_id": "Kijai/LTX2.3_comfy",
     "filename": "diffusion_models/ltx-2.3-22b-distilled-1.1_transformer_only_fp8_scaled.safetensors",
     "final_dir": UNET_DIR, "flatten": True},

    # --- 2. 空间放大模型 (LatentUpscaleModelLoader, Stage #2 Upscale 组) ---
    {"repo_id": "Lightricks/LTX-2.3",
     "filename": "ltx-2.3-spatial-upscaler-x2-1.1.safetensors",
     "final_dir": "/content/ComfyUI/models/latent_upscale_models"},

    # --- 3. 文本编码器 + 投影矩阵 (DualCLIPLoader) ---
    # fp4_mixed 需要 Blackwell, 正好匹配 G4 运行时
    {"repo_id": "Comfy-Org/ltx-2",
     "filename": "split_files/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors",
     "final_dir": "/content/ComfyUI/models/text_encoders"},
    {"repo_id": "Kijai/LTX2.3_comfy",
     "filename": "text_encoders/ltx-2.3_text_projection_bf16.safetensors",
     "final_dir": "/content/ComfyUI/models/text_encoders", "flatten": True},

    # --- 4. 视频 / 音频 / 快速预览 VAE (VAELoader x2 + VAELoaderKJ) ---
    {"repo_id": "Kijai/LTX2.3_comfy", "filename": "vae/LTX23_video_vae_bf16.safetensors",
     "final_dir": "/content/ComfyUI/models/vae", "flatten": True},
    {"repo_id": "Kijai/LTX2.3_comfy", "filename": "vae/LTX23_audio_vae_bf16.safetensors",
     "final_dir": "/content/ComfyUI/models/vae", "flatten": True},
    {"repo_id": "Kijai/LTX2.3_comfy", "filename": "vae/taeltx2_3.safetensors",
     "final_dir": "/content/ComfyUI/models/vae", "flatten": True},

    # ==========================================
    # 以下模型本工作流未使用, 已全部注释
    # ==========================================
    # ❌ 完整 22b 大模型: 工作流用 UNETLoader 加载 transformer, 没有 CheckpointLoaderSimple
    # {"repo_id": "Lightricks/LTX-2.3-fp8", "filename": "ltx-2.3-22b-dev-fp8.safetensors", "final_dir": "/content/ComfyUI/models/checkpoints"},
    # ❌ 蒸馏 LoRA: 主模型已是 distilled 合并权重, 工作流内无任何 LoraLoader 节点
    # {"repo_id": "Kijai/LTX2.3_comfy", "filename": "loras/ltx-2.3-22b-distilled-lora-dynamic_fro09_avg_rank_105_bf16.safetensors", "final_dir": "/content/ComfyUI/models/loras", "flatten": True},
    # {"repo_id": "Lightricks/LTX-2.3", "filename": "ltx-2.3-22b-distilled-lora-384-1.1.safetensors", "final_dir": "/content/ComfyUI/models/loras"},
    # {"repo_id": "Comfy-Org/ltx-2", "filename": "split_files/text_encoders/gemma_3_12B_it_fp8_scaled.safetensors", "final_dir": "/content/ComfyUI/models/text_encoders"},
    # {"repo_id": "Comfy-Org/ltx-2", "filename": "split_files/text_encoders/gemma_3_12B_it.safetensors", "final_dir": "/content/ComfyUI/models/text_encoders"},
    # {"repo_id": "Comfy-Org/ltx-2.3", "filename": "split_files/loras/ltx-2.3-id-lora-talkvid-3k.safetensors", "final_dir": "/content/ComfyUI/models/loras"},
    # {"repo_id": "Lightricks/LTX-2.3-22b-IC-LoRA-Union-Control", "filename": "ltx-2.3-22b-ic-lora-union-control-ref0.5.safetensors", "final_dir": "/content/ComfyUI/models/loras"},
    # {"repo_id": "Lightricks/LTX-2.3", "filename": "ltx-2.3-22b-distilled-1.1.safetensors", "final_dir": "/content/ComfyUI/models/checkpoints"},
    # {"repo_id": "stronman/LTX-2.3-Transition-LORA", "filename": "ltx2.3-transition.safetensors", "final_dir": "/content/ComfyUI/models/loras"},
    # {"repo_id": "stronman/Ltx2.3-VBVR-lora-I2V", "filename": "Ltx2.3-Licon-VBVR-I2V-390K-R32.safetensors", "final_dir": "/content/ComfyUI/models/loras"},
    # {"repo_id": "Comfy-Org/ltx-2.3", "filename": "split_files/loras/ltx_2.3_22b_distilled_1.1_lora_dynamic_fro09_avg_rank_111_bf16.safetensors", "final_dir": "/content/ComfyUI/models/loras", "flatten": True},
    # {"repo_id": "Comfy-Org/ltx-2", "filename": "split_files/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors", "final_dir": "/content/ComfyUI/models/loras", "flatten": True},
    # {"repo_id": "Lightricks/LTX-2.3-22b-IC-LoRA-HDR", "filename": "ltx-2.3-22b-ic-lora-hdr-0.9.safetensors", "final_dir": "/content/ComfyUI/models/loras"},
    # {"repo_id": "Lightricks/LTX-2.3-22b-IC-LoRA-Motion-Track-Control", "filename": "ltx-2.3-22b-ic-lora-motion-track-control-ref0.5.safetensors", "final_dir": "/content/ComfyUI/models/loras"},
    # {"repo_id": "Lightricks/LTX-2.3-22b-IC-LoRA-LipDub", "filename": "ltx-2.3-22b-ic-lora-lipdub-0.9.safetensors", "final_dir": "/content/ComfyUI/models/loras"},
    # {"repo_id": "h94/IP-Adapter-FaceID", "filename": "ip-adapter-faceid-plusv2_sdxl.bin", "final_dir": "/content/ComfyUI/models/ipadapter"},
    # {"repo_id": "h94/IP-Adapter", "filename": "models/image_encoder/model.safetensors", "final_dir": "/content/ComfyUI/models/clip_vision", "flatten": True, "rename_to": "CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors"},
    # {"repo_id": "joyfox/LTX2.3-ICEdit-Insight", "filename": "ltx2.3-ic-watermark-remove-general.safetensors", "final_dir": "/content/ComfyUI/models/loras", "rename_to": "ltx2.3-ic-watermark-remove-general-lora.safetensors"},
    # {"repo_id": "joyfox/LTX2.3-ICEdit-Insight", "filename": "ltx2.3-ic-subtitles-remove-general.safetensors", "final_dir": "/content/ComfyUI/models/loras", "rename_to": "ltx2.3-ic-subtitles-remove-general-lora.safetensors"},
    # {"repo_id": "joyfox/LTX2.3-ICEdit-Insight", "filename": "ltx2.3-video-restoration-general.safetensors", "final_dir": "/content/ComfyUI/models/loras", "rename_to": "ltx2.3-video-restoration-general-lora.safetensors"},
    # {"repo_id": "joyfox/LTX2.3-ICEdit-Insight", "filename": "ltx2.3-ic-video-upscale-general.safetensors", "final_dir": "/content/ComfyUI/models/loras", "rename_to": "ltx2.3-ic-video-upscale-general-lora.safetensors"},
    # {"repo_id": "Muapi/ltx2.3-resolution-enhancement-guofeng", "filename": "ltx2.3-resolution-enhancement-guofeng.safetensors", "final_dir": "/content/ComfyUI/models/loras", "rename_to": "LTX-2.3-Guofeng-V0.1.safetensors"},
]


def download_model(task):
    try:
        os.makedirs(task["final_dir"], exist_ok=True)
        base_name = task.get("rename_to", os.path.basename(task["filename"]))
        final_path = os.path.join(task["final_dir"], base_name)

        if os.path.exists(final_path):
            print(f"⏩ 已存在, 跳过下载: {base_name}")
            return

        downloaded_path = hf_hub_download(
            repo_id=task["repo_id"],
            filename=task["filename"],
            local_dir=task["final_dir"],
            repo_type="model",
        )

        if ("flatten" in task or "rename_to" in task) and downloaded_path != final_path:
            os.makedirs(os.path.dirname(final_path), exist_ok=True)
            if os.path.exists(final_path):
                os.remove(final_path)
            shutil.move(downloaded_path, final_path)

        print(f"✅ 成功就绪: {base_name}")
    except Exception as e:
        print(f"❌ 失败 {task['filename'].split('/')[-1]}: {e}")


print(f"🚀 开始并发下载本工作流所需的 {len(downloads)} 个模型...")
with ThreadPoolExecutor(max_workers=4) as executor:
    list(executor.map(download_model, downloads))

# 兼容旧版 ComfyUI: 在 models/unet 下做一份硬链接
legacy_dir = "/content/ComfyUI/models/unet"
os.makedirs(legacy_dir, exist_ok=True)
for f in os.listdir(UNET_DIR):
    src, dst = os.path.join(UNET_DIR, f), os.path.join(legacy_dir, f)
    if not os.path.exists(dst):
        try:
            os.link(src, dst)
        except OSError:
            pass

print("\n🎉 模型下载完毕")


In [ ]:
# ==========================================
# Cell 5: 下载工作流 JSON 到 ComfyUI 的 workflows 目录
# ==========================================
import os
import urllib.request

wf_dir = "/content/ComfyUI/user/default/workflows"
os.makedirs(wf_dir, exist_ok=True)

wf_url = ("https://raw.githubusercontent.com/WhatDreamsCost/WhatDreamsCost-ComfyUI/"
          "main/example_workflows/LTX_Director_2_Workflow_Distilled.json")
wf_path = os.path.join(wf_dir, "LTX_Director_2_Workflow_Distilled.json")

try:
    urllib.request.urlretrieve(wf_url, wf_path)
    print(f"✅ 工作流已就绪: {wf_path}")
    print("👉 启动 ComfyUI 后在左侧 Workflows 面板直接打开")
except Exception as e:
    print(f"❌ 下载失败: {e}")

# ❌ GGUF 版本: 面向 8-12GB 小显存卡, 在 G4 (96GB) 上反而更慢且有损, 不下载
# wf_url_gguf = ("https://raw.githubusercontent.com/WhatDreamsCost/WhatDreamsCost-ComfyUI/"
#                "main/example_workflows/LTX_Director_2_Workflow_GGUF.json")


In [ ]:
# ==========================================
# Cell 6: FRP 内网穿透配置 (流式解压极速部署)
# ==========================================
import os
from google.colab import userdata

try:
    VPS_IP = userdata.get('VPS_IP')
    FRP_TOKEN = userdata.get('FRP_TOKEN')
except:
    raise Exception("❌ 请检查 'VPS_IP' 和 'FRP_TOKEN' 密钥是否配置！")

# 极速流式下载解压
if not os.path.exists("/content/frp_0.56.0_linux_amd64"):
    !wget -qO- https://github.com/fatedier/frp/releases/download/v0.56.0/frp_0.56.0_linux_amd64.tar.gz | tar -xz -C /content

frpc_conf = f"""
serverAddr = "{VPS_IP}"
serverPort = 7000
auth.token = "{FRP_TOKEN}"

[[proxies]]
name = "comfyui_web_colab"
type = "tcp"
localIP = "127.0.0.1"
localPort = 8188
remotePort = 8090
"""
with open("/content/frp_0.56.0_linux_amd64/frpc.toml", "w") as f:
    f.write(frpc_conf.strip())

print("✅ FRP 极速部署并配置完毕！")

In [ ]:
# ==========================================
# Cell 7: Google API Key 注入 —— ❌ 本工作流未使用, 整段已注释
# ------------------------------------------
# 原因: 这段是为 comfyui-google-genmedia-custom-nodes (Veo / Imagen 等云端节点) 准备的。
#       本工作流的文本编码器是本地 gemma_3_12B (DualCLIPLoader), 不调用任何 Google API。
# 如果你后续要加云端节点, 取消下面注释即可。
# ==========================================

# import os
# from google.colab import userdata
#
# try:
#     api_key = userdata.get('GOOGLE_API_KEY')
#     os.environ["GOOGLE_API_KEY"] = api_key
#     os.environ["GEMINI_API_KEY"] = api_key
#     # 切断 Vertex AI 默认 IAM 证书链, 逼 SDK 降级使用 API_KEY
#     os.environ["GCLOUD_PROJECT"] = ""
#     os.environ["NO_GCE_CHECK"] = "True"
#     os.environ["GOOGLE_AUTH_SUPPRESS_CREDENTIALS_WARNINGS"] = "1"
#     print("✅ 认证拦截补丁已成功注入！")
# except userdata.SecretNotFoundError:
#     print("❌ 未在 Colab Secrets 中找到 'GOOGLE_API_KEY'。")

print("⏩ 已跳过 Google API 认证补丁 (本工作流不需要)")


In [ ]:
# ==========================================
# Cell 8: 启动 FRP 和 ComfyUI (重启界面时运行),不注册，启动快！访问域名
# ==========================================
import subprocess
import threading
import os
import configparser

import time

os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
# --- 新增：防断开保活机制 ---
def keep_alive():
    while True:
        time.sleep(300)  # 每 5 分钟
        print("\n[Keep-Alive] 保持 Colab 连接活跃中...")

print("⏳ 正在启动防断开后台保活线程...")
threading.Thread(target=keep_alive, daemon=True).start()
# --------------------------

# 1. 在后台新线程启动 FRP
def start_frpc():
    subprocess.run(["/content/frp_0.56.0_linux_amd64/frpc", "-c", "/content/frp_0.56.0_linux_amd64/frpc.toml"])

print("⏳ 正在后台唤起 FRP 穿透服务...")
threading.Thread(target=start_frpc, daemon=True).start()

print("============================================================")
print("✅ FRP 穿透已就绪！")
# 这里已修改为你的指定域名（保留了 8080 端口，如果你的服务端用 Nginx 做了 80 端口反代，可以把 :8080 删掉）
print("👉 启动完成后，请通过浏览器访问: http://cjp.usdream.dpdns.org:8090")
print("============================================================\n")

# ---------------------------------------------------------
# 新增：彻底拦截 ComfyUI-Manager 启动时的联网 Fetch 行为
# ---------------------------------------------------------
print("⏳ 正在配置 Manager 网络模式以跳过 Fetch...")
# 兼容所有版本的配置文件路径 (特别是最新的 __manager 路径)
manager_config_paths = [
    "/content/ComfyUI/user/__manager/config.ini",               # 最新版 V3.39+ 配置路径
    "/content/ComfyUI/user/default/ComfyUI-Manager/config.ini", # 较新版配置路径
    "/content/ComfyUI/custom_nodes/ComfyUI-Manager/config.ini"  # 老旧版配置路径
]

for config_path in manager_config_paths:
    os.makedirs(os.path.dirname(config_path), exist_ok=True)
    config = configparser.ConfigParser()

    if os.path.exists(config_path):
        config.read(config_path)

    if 'default' not in config:
        config['default'] = {}

    # 将网络模式改为 private
    config['default']['network_mode'] = 'private'

    with open(config_path, 'w') as f:
        config.write(f)

print("✅ 已成功切断 Fetch ComfyRegistry Data 流程！\n")
# ---------------------------------------------------------

# 2. 启动 ComfyUI 主程序
print("⏳ 正在启动 ComfyUI 主进程...\n")
%cd /content/ComfyUI
!python main.py --dont-print-server